# Fine-tune Qwen3 on RBI Regulatory QA Dataset (QLoRA) — 6GB Laptop GPU Edition

**Dataset**: `Magneto/qa_with_personas` — 23,892 Q&A pairs generated from RBI (Reserve Bank of India) circulars and master directions. Despite the dataset card saying "BI", this is RBI data (find-and-replace typo in the original README).

**Format**: RAG-style — `question + context -> answer`. The model learns to answer using a provided regulatory passage, which is the right setup since you'll eventually pair this with a retriever (or just always supply the relevant context at inference time).

**Target model**: `Qwen/Qwen3-4B`, with automatic fallback to `Qwen/Qwen3-1.7B` if you OOM. Both are QLoRA (4-bit NF4 + LoRA adapters). Training always runs in `bf16` regardless of GPU — Qwen3's native parameters are bf16, and using `fp16` instead crashes (PyTorch's fp16 `GradScaler` doesn't support bf16 gradients).

**This version is tuned for a 6GB-VRAM laptop GPU** (e.g. RTX 3050/3050 Ti/4050 mobile, GTX 1660, etc.) running locally in Jupyter instead of Colab. The GPU-detection cell measures your actual VRAM and — for anything at or under ~8GB — automatically picks the conservative settings that the cloud version of this notebook only mentioned as "try this if you OOM": fewer LoRA target modules, a shorter sequence length, micro-batch of 1, and gradient checkpointing. Everything else (dataset, formatting, QLoRA setup, training loop, logging, saving, evaluation) is unchanged from the original.

Run cells top to bottom. The GPU-detection cell sets sensible defaults automatically; the config cell right after it is the one you'll touch most if you want to override them.


## 1. Install dependencies

Pinned versions that are known to work together with Qwen3 + QLoRA as of mid-2026.

**Local-GPU note (read before running)**: unlike Colab, your laptop's `torch` install needs to already
match your NVIDIA driver's CUDA version, or `torch.cuda.is_available()` will silently come back `False`
and everything will fall back to CPU (which will not realistically finish training). Run `!nvidia-smi` in
the cell below first to see your driver's max supported CUDA version, and if `import torch; torch.cuda.is_available()`
prints `False` after the install, reinstall torch from https://pytorch.org/get-started/locally/ with the
matching CUDA wheel *before* re-running anything else here.

**Windows users**: `bitsandbytes` 4-bit support on native Windows has historically lagged Linux. If you hit
import errors or silent fallbacks to CPU-only quantization, run this notebook under **WSL2** instead of
native Windows Python — it's the path of least resistance for QLoRA on Windows hardware.


In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate peft bitsandbytes trl datasets evaluate sentencepiece tensorboard matplotlib wandb
!pip install -q -U flash-attn --no-build-isolation || echo "flash-attn install failed/skipped — not required, just faster if it works"

In [ ]:
!nvidia-smi


In [ ]:
import torch, transformers, peft, trl, bitsandbytes
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

### Weights & Biases login

Paste your W&B API key when prompted (find it at https://wandb.ai/authorize). `getpass` hides the
characters as you type and — importantly — does NOT save the key anywhere in this notebook file, so it's
safe to share/upload the `.ipynb` afterward without leaking your key. You only need to do this once per
Colab session; it'll stay logged in until the runtime restarts.

In [ ]:
import wandb
from getpass import getpass

wandb_api_key = getpass("Paste your W&B API key (input hidden): ")
wandb.login(key=wandb_api_key)

# Clear the key from this variable immediately after use — it's no longer needed,
# and there's no reason to keep a copy of the secret sitting in a live notebook variable.
del wandb_api_key
print("W&B login successful.")

### Detect GPU and set platform-appropriate defaults

This cell measures your **actual VRAM** via `torch.cuda.get_device_properties` rather than guessing from the
GPU name (a 6GB laptop 3050 and a 24GB desktop 3090 both report "RTX 3090"-style names inconsistently across
drivers, so a name-string match isn't reliable — total memory in bytes is).

For anything at or under ~8GB (which covers most 6GB laptop GPUs once you subtract the ~0.3-0.5GB the OS/
display/CUDA context already uses), we automatically tighten three things that the memory math actually
depends on:
- **`LORA_TARGET_MODULES`**: only `q_proj`/`v_proj` instead of all four attention projections — fewer adapter
  matrices, less optimizer state, less activation memory.
- **`MAX_SEQ_LENGTH` suggestion**: 384 instead of 512 — memory scales with sequence length.
- **Micro-batch / gradient checkpointing**: micro-batch of 1 with checkpointing on, same as before.

**Note on precision**: Qwen3's native parameters are bf16, and mixing them with PyTorch's fp16 `GradScaler`
crashes outright (`NotImplementedError: ..._unscale_cuda not implemented for 'BFloat16'`). bf16 training is
required here regardless of GPU generation — this isn't a tunable setting for Qwen3.


In [ ]:
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
_vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0.0

# 8GB is a deliberately generous cutoff: it catches every 6GB laptop GPU (3050, 3050 Ti, 4050,
# GTX 1660, etc.) plus a little headroom, without falsely triggering on real 8GB+ cards.
_is_low_vram = (not torch.cuda.is_available()) or _vram_gb <= 8.0

# Always bf16: Qwen3's native dtype is bf16, and fp16 + GradScaler is incompatible with it
# (crashes in torch.amp.grad_scaler regardless of GPU generation). Not a tunable choice here.
USE_BF16 = True

# Gradient checkpointing stays on regardless of VRAM size: the SFT chunked_nll loss path is
# memory-safe on its own, but TRL also computes a per-token entropy metric for logging that —
# on a non-chunked fallback path — can materialize a full [batch*seq_len, vocab_size] tensor
# (~150k vocab for Qwen3). Checkpointing buys headroom against that on any GPU.
USE_GRAD_CHECKPOINTING = True
SUGGESTED_MICRO_BATCH = 1

if _is_low_vram:
    SUGGESTED_LORA_TARGET_MODULES = ["q_proj", "v_proj"]
    SUGGESTED_MAX_SEQ_LENGTH = 384
else:
    SUGGESTED_LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
    SUGGESTED_MAX_SEQ_LENGTH = 512

print(f"Detected GPU: {_gpu_name or 'none'}  (VRAM: {_vram_gb:.1f} GB)")
print(f"Low-VRAM mode: {_is_low_vram}")
print(f"USE_BF16={USE_BF16}  USE_GRAD_CHECKPOINTING={USE_GRAD_CHECKPOINTING}  "
      f"SUGGESTED_MICRO_BATCH={SUGGESTED_MICRO_BATCH}")
print(f"SUGGESTED_LORA_TARGET_MODULES={SUGGESTED_LORA_TARGET_MODULES}  "
      f"SUGGESTED_MAX_SEQ_LENGTH={SUGGESTED_MAX_SEQ_LENGTH}")
print("These feed into the config cell below — override there if you know better for your setup.")


## 2. Config

Set `MODEL_NAME` to try 4B first. If you hit `CUDA out of memory` during training, just change this one line to the 1.7B fallback and re-run from here — no other code changes needed.

`LORA_TARGET_MODULES` and `MAX_SEQ_LENGTH` below default to whatever the GPU-detection cell suggested, which on a 6GB card means `["q_proj", "v_proj"]` and `384`. You can still override either by hand.

**Sized down for a fast smoke test**: `TRAIN_SUBSET_SIZE=200`, 1 epoch — together with the seq length above this keeps total steps small, so you can confirm the whole pipeline (load, LoRA, train, save, generate) actually completes on your laptop before committing to a longer run. Once this finishes cleanly, bump `TRAIN_SUBSET_SIZE` up (or to `None` for the full 19,113-example train set), restore `NUM_EPOCHS=2`, and re-check `MAX_SEQ_LENGTH` against Cell 3a's real percentiles.


In [ ]:
# ============================================================
# MODEL SELECTION
# ============================================================
# We try the bigger 4B model first (better quality, more knowledge capacity), and fall back
# to the smaller 1.7B model automatically if the GPU runs out of memory loading it (see the
# model-loading cell later, which wraps this in a try/except). On a 6GB card, 4B in 4-bit NF4
# is roughly 2.5-3GB of weights alone, which usually leaves enough room for LoRA + activations
# at MAX_SEQ_LENGTH=384 with micro-batch 1 — but the fallback is there in case your specific
# card/driver/background-app overhead doesn't leave quite enough headroom.
MODEL_NAME = "Qwen/Qwen3-4B"             # ~4 billion parameters
FALLBACK_MODEL_NAME = "Qwen/Qwen3-1.7B"  # ~1.7 billion parameters — smaller, fits in less VRAM

# --- Dataset ---
DATASET_NAME = "Magneto/qa_with_personas"  # the RBI regulatory Q&A dataset on Hugging Face Hub

# ============================================================
# SEQUENCE LENGTH
# ============================================================
# MAX_SEQ_LENGTH = the maximum number of TOKENS (roughly, word-fragments — not characters) the
# model will see per training example, after combining the system prompt + context + question +
# answer. Longer sequences cost more GPU memory (memory scales with sequence length), so this is
# a direct memory/quality tradeoff:
#   - Too SHORT  -> long contexts/answers get cut off (truncated), and the model never learns
#                   from the missing part. Bad for quality.
#   - Too LONG   -> wastes GPU memory padding short examples up to this length, and risks OOM.
# Defaults to the GPU-detection cell's suggestion (384 on a 6GB card). Cell 3a below measures the
# REAL token-length distribution of this dataset — after running it, come back and check this
# against its p95 value; on a 6GB card you'll likely want to accept some truncation on the long
# tail rather than raise this further.
MAX_SEQ_LENGTH = SUGGESTED_MAX_SEQ_LENGTH

# ============================================================
# LoRA CONFIG (the "adapter" — what actually gets trained)
# ============================================================
# LoRA = Low-Rank Adaptation. Instead of updating all ~4 billion parameters of the base model
# (which would need far more VRAM than we have), we freeze the base model entirely and insert
# small trainable "adapter" matrices into specific layers. Only those small matrices get updated
# during training — typically well under 1% of the total parameter count.

# LORA_R ("rank"): controls the SIZE of each adapter matrix, and therefore how much new
# information the adapter can encode. Think of it like the resolution of the adjustment — rank 8
# is a small, common starting point. Higher rank (16, 32, 64...) lets the adapter learn more
# nuanced patterns but uses more memory and risks overfitting on a smaller dataset.
LORA_R = 8

# LORA_ALPHA: a scaling factor applied to the adapter's output before it's added back into the
# frozen base model's computation. Conventionally set to 2x the rank (8 -> 16) as a starting
# heuristic; this keeps the adapter's influence at a reasonable magnitude without needing to
# separately retune the learning rate every time you change LORA_R.
LORA_ALPHA = 16

# LORA_DROPOUT: during training, randomly "turns off" this fraction (5%) of the adapter's
# internal connections on each forward pass. This is a regularization technique — it prevents
# the small adapter from memorizing the training data too rigidly (overfitting), at a very
# slight cost to how fast it learns.
LORA_DROPOUT = 0.05

# LORA_TARGET_MODULES: WHICH weight matrices inside each transformer layer get an adapter.
# q_proj/k_proj/v_proj/o_proj are the four projection matrices inside the attention mechanism
# (Query, Key, Value, Output). Defaults to the GPU-detection cell's suggestion — on a 6GB card
# that's just q_proj/v_proj, which is usually still enough to meaningfully steer the model's
# behavior while keeping the adapter (and its optimizer state) as small as possible. If you have
# headroom left after a successful run (check GPU memory in the training-loop output), uncomment
# the line below to add k_proj/o_proj back, or even the MLP layers, for more capacity.
LORA_TARGET_MODULES = list(SUGGESTED_LORA_TARGET_MODULES)
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # uncomment if you have headroom
# LORA_TARGET_MODULES += ["gate_proj", "up_proj", "down_proj"]    # uncomment for even more capacity

# ============================================================
# TRAINING CONFIG
# ============================================================
OUTPUT_DIR = "./qwen3-rbi-qa-lora"  # where checkpoints, logs, and the final adapter get saved

# PER_DEVICE_BATCH_SIZE ("micro-batch"): how many training examples are processed together in
# ONE forward+backward pass on the GPU. This is the number that most directly determines whether
# you OOM — bigger batches need proportionally more memory. SUGGESTED_MICRO_BATCH comes from the
# GPU-detection cell above (1, on a 6GB card — don't raise this without a lot of headroom).
PER_DEVICE_BATCH_SIZE = SUGGESTED_MICRO_BATCH

# GRAD_ACCUM_STEPS ("gradient accumulation steps"): since we can't fit a large batch in memory
# at once, we instead run several small micro-batches in a row, add up ("accumulate") their
# gradients, and only THEN update the model weights once. This achieves the same training
# behavior as a true larger batch, at the memory cost of only the small micro-batch.
# EFFECTIVE BATCH SIZE = PER_DEVICE_BATCH_SIZE x GRAD_ACCUM_STEPS. We target ~16 here regardless
# of what micro-batch size your GPU got assigned, by dividing 16 by the micro-batch.
GRAD_ACCUM_STEPS = 16 // PER_DEVICE_BATCH_SIZE

# NUM_EPOCHS: how many full passes over the (sub-sampled, for the smoke test) training set to
# make. 1 epoch = the model sees every training example exactly once. More epochs = more
# learning, but also more risk of overfitting (memorizing the training data instead of
# generalizing) on a small dataset. Currently 1 for a fast smoke test; bump to 2-3 for a real run.
NUM_EPOCHS = 1

# LEARNING_RATE: how big a step the optimizer takes when updating the LoRA adapter weights after
# each accumulated batch. 2e-4 (0.0002) is a fairly standard LoRA learning rate — notably HIGHER
# than you'd use for full fine-tuning (often 1e-5 to 5e-5), because we're only updating a tiny
# fraction of parameters and need bigger steps to move them meaningfully. Too high -> unstable/
# diverging loss; too low -> learns too slowly to see results in a short run.
LEARNING_RATE = 2e-4

# WARMUP_RATIO: for the first 3% of total training steps, the learning rate ramps up LINEARLY
# from 0 to LEARNING_RATE, instead of starting at full strength immediately. This avoids a
# potentially destabilizing large update on the very first few batches, before the optimizer's
# internal statistics have had a chance to initialize properly.
WARMUP_RATIO = 0.03

# LOGGING_STEPS: how often (in training steps) to print/record a loss value. Every 5 steps here
# means you'll see a new "Training Loss" row roughly every 5 optimizer updates — small enough to
# give you near-immediate feedback on a short smoke test, without flooding the output on a long run.
LOGGING_STEPS = 5

# SAVE_STEPS: how often to write a model checkpoint to disk. EVAL_STEPS: how often to run a pass
# over the held-out validation set and compute "Validation Loss" (a measure of how well the
# model generalizes to data it wasn't trained on, as opposed to training loss which can look
# good even if the model is just memorizing).
SAVE_STEPS = 50
EVAL_STEPS = 50

# ============================================================
# SMOKE-TEST SUBSAMPLING
# ============================================================
# Rather than risk a long, expensive run failing halfway through on a real bug, we first run on
# a small slice of data to confirm the whole pipeline works end-to-end (loads, trains, saves,
# generates) quickly. TRAIN_SUBSET_SIZE=200 with the settings above gives roughly 200/16 ≈ 13
# training steps total — fast enough to finish in minutes even on a laptop GPU. Set to None to
# use the FULL dataset (19,113 training examples) once you've verified everything works — on a
# 6GB card at the full dataset, expect this to take meaningfully longer than on a cloud T4/L4.
TRAIN_SUBSET_SIZE = 200
EVAL_SUBSET_SIZE = 50

# ============================================================
# EXPERIMENT TRACKING (Weights & Biases + TensorBoard)
# ============================================================
# WANDB_PROJECT: groups all your runs together under one project name on wandb.ai, so you can
# compare different attempts (different models, hyperparameters, dataset sizes) side by side on
# the same dashboard instead of them appearing as unrelated/anonymous runs.
WANDB_PROJECT = "qwen3-rbi-qa-finetune"

# RUN_NAME: a human-readable label for THIS specific run, shown in both W&B and TensorBoard. We
# build it from the model name, subset size, and epoch count so that — when you look back later
# at a list of 10 different runs — you can tell which settings each one used just from the name,
# without having to dig through configs.
import time as _time
RUN_NAME = (
    f"{MODEL_NAME.split('/')[-1]}"
    f"_subset{TRAIN_SUBSET_SIZE if TRAIN_SUBSET_SIZE else 'FULL'}"
    f"_ep{NUM_EPOCHS}"
    f"_{_time.strftime('%Y%m%d-%H%M%S')}"
)
print("This run will be logged as:", RUN_NAME)


## 3. Load and inspect the dataset

`load_dataset` caches downloaded parquet files under `~/.cache/huggingface/datasets/` on your laptop's local
disk. Unlike Colab, this persists across sessions on its own (your laptop's disk doesn't get wiped on
restart), so you'll only download this dataset once.


In [ ]:
# (Colab's Google-Drive cache-mounting step from the cloud version of this notebook is not
# needed locally — your laptop's Hugging Face datasets cache at ~/.cache/huggingface/datasets/
# already persists across restarts on its own.)
print("Running locally — using the default local Hugging Face datasets cache.")


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print()
print("Example record:")
print(raw_dataset["train"][0])

### 3a. Check token length distribution before fixing MAX_SEQ_LENGTH

This matters because the dataset card only tells us `context` is ~1000 chars — we should verify actual tokenized length (question + context + answer) before committing to a sequence length, otherwise we either waste VRAM on padding or silently truncate answers.

In [ ]:
from transformers import AutoTokenizer
import numpy as np

_tok_probe = AutoTokenizer.from_pretrained(MODEL_NAME)

sample = raw_dataset["train"].shuffle(seed=42).select(range(min(1000, len(raw_dataset["train"]))))
lengths = []
for ex in sample:
    full_text = f"{ex['context']}\n\n{ex['question']}\n\n{ex['answer']}"
    lengths.append(len(_tok_probe(full_text)["input_ids"]))

lengths = np.array(lengths)
print(f"mean={lengths.mean():.0f}  median={np.median(lengths):.0f}  "
      f"p90={np.percentile(lengths, 90):.0f}  p95={np.percentile(lengths, 95):.0f}  "
      f"p99={np.percentile(lengths, 99):.0f}  max={lengths.max()}")
print()
print("If p95 is well under MAX_SEQ_LENGTH, you're safe. If not, either raise "
      "MAX_SEQ_LENGTH (costs VRAM) or accept truncation on the longest tail examples.")

## 4. Format for RAG-style instruction tuning

Each example becomes a chat-formatted sequence: a system prompt establishing the assistant's role, a user turn with the regulatory context + question, and an assistant turn with the answer. We use Qwen3's chat template with `enable_thinking=False` since this is direct QA, not a reasoning task — we don't want the model generating `<think>` blocks for straightforward regulatory lookups.

In [ ]:
SYSTEM_PROMPT = (
    "You are a knowledgeable assistant on Reserve Bank of India (RBI) banking regulations. "
    "Answer the question using only the information in the provided regulatory context. "
    "Be precise and comprehensive, and note when the context does not fully answer the question."
)

def build_messages(example):
    user_content = f"Context:\n{example['context']}\n\nQuestion: {example['question']}"
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

train_ds = raw_dataset["train"].map(build_messages, remove_columns=raw_dataset["train"].column_names)
eval_ds = raw_dataset["validation"].map(build_messages, remove_columns=raw_dataset["validation"].column_names)

if TRAIN_SUBSET_SIZE is not None:
    train_ds = train_ds.shuffle(seed=42).select(range(min(TRAIN_SUBSET_SIZE, len(train_ds))))
if EVAL_SUBSET_SIZE is not None:
    eval_ds = eval_ds.shuffle(seed=42).select(range(min(EVAL_SUBSET_SIZE, len(eval_ds))))

print(f"train: {len(train_ds)}  |  eval: {len(eval_ds)}")
print()
print("Example formatted record:")
print(train_ds[0])

## 5. Load model in 4-bit (QLoRA) with automatic fallback

Tries `MODEL_NAME` (4B) first. If CUDA OOMs during model load itself (rare — load is cheaper than training), it falls back to the 1.7B model automatically. **OOM during the actual training loop later is more likely** — if that happens, just re-run from Cell 2 (config) with `MODEL_NAME = FALLBACK_MODEL_NAME` and re-execute downstream cells.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

_compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=_compute_dtype,
    bnb_4bit_use_double_quant=True,
)

def load_model_and_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=_compute_dtype,
    )
    model.config.use_cache = False  # required for gradient checkpointing; re-enabled before generation in Cell 10
    return model, tok

active_model_name = MODEL_NAME
try:
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
    print(f"Loaded {MODEL_NAME} successfully.")
except torch.cuda.OutOfMemoryError:
    print(f"OOM loading {MODEL_NAME}. Falling back to {FALLBACK_MODEL_NAME}...")
    torch.cuda.empty_cache()
    active_model_name = FALLBACK_MODEL_NAME
    model, tokenizer = load_model_and_tokenizer(FALLBACK_MODEL_NAME)
    print(f"Loaded {FALLBACK_MODEL_NAME} successfully.")

print("Active model:", active_model_name)
print("GPU memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))

## 6. Prepare for k-bit training and define the LoRA config

**Important**: we deliberately do NOT call `get_peft_model()` here. `SFTTrainer` applies its memory-efficient
chunked cross-entropy patch (`loss_type="chunked_nll"`, the default) by patching `.forward()` on the inner
causal LM. For that patch to land correctly on a LoRA-wrapped model, TRL needs to do the PEFT-wrapping itself
(via `peft_config=` below) — wrapping the model ourselves first and handing `SFTTrainer` an already-PeftModel
instance can cause the chunked patch to silently miss, which is what produced the earlier
`entropy_from_logits` OOM: the model fell back to computing full, unchunked `[batch*seq, vocab]` logits.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=USE_GRAD_CHECKPOINTING)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,  # do NOT include "lm_head" here — incompatible with loss_type="chunked_nll"
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA config ready (not yet applied). SFTTrainer will wrap the model with this in the next step.")

## 7. Tokenize with chat template (thinking mode OFF)

In [ ]:
def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return text

# Sanity check on one example
print(formatting_func(train_ds[0])[:1500])

## 8. Train with TRL's SFTTrainer

`paged_adamw_8bit` is the key setting that makes this fit on small VRAM — it streams optimizer states to CPU when GPU memory is tight, at a small speed cost. Don't swap this for plain `adamw_torch` on a 6GB card.

**Note**: newer TRL versions renamed `SFTConfig`'s `max_seq_length` argument to `max_length` (the old name was deprecated, then removed). This notebook uses `max_length`. If you're on an older pinned TRL (<0.20) and get an unrecognized-argument error in the opposite direction, swap it back to `max_seq_length`. Run `import trl; print(trl.__version__)` if you want to check which one applies.

In [ ]:
from trl import SFTTrainer, SFTConfig
import os

# WANDB_PROJECT is read automatically by transformers/accelerate from this environment variable —
# it's how the project name set in the config cell actually reaches W&B's servers.
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,                              # checkpoints/logs saved here

    # --- Batch sizing (see config cell for full explanation of micro-batch vs effective batch) ---
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # --- Core training hyperparameters ---
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    # cosine schedule: the learning rate follows a smooth downward curve (shaped like one hump
    # of a cosine wave) from its peak down to ~0 by the end of training, rather than staying flat
    # the whole time. This tends to produce slightly better final results than a constant rate,
    # since the model takes smaller, more careful steps as it gets closer to convergence.
    lr_scheduler_type="cosine",

    # --- How often to log / checkpoint / evaluate (see config cell for exact meanings) ---
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",     # evaluate every EVAL_STEPS steps (not just once per epoch)
    save_strategy="steps",     # checkpoint every SAVE_STEPS steps
    save_total_limit=2,        # only keep the 2 most recent checkpoints on disk (saves space)

    # --- Memory-saving settings (important for fitting on limited VRAM) ---
    # paged_adamw_8bit: a memory-efficient version of the Adam optimizer. Normal Adam keeps two
    # extra numbers (momentum + variance) PER PARAMETER, which doubles memory usage on top of the
    # parameters themselves. This variant stores those extra numbers in 8-bit instead of 32-bit,
    # AND can temporarily move them to CPU RAM ("paging") if GPU memory gets tight — similar to
    # how an OS pages memory to disk under pressure. This is the single most important setting
    # for fitting training on a small GPU.
    optim="paged_adamw_8bit",

    # bf16/fp16: which reduced-precision number format to train in (instead of full 32-bit floats,
    # which would use 2x the memory). USE_BF16 is decided automatically earlier in the notebook —
    # currently always True, because Qwen3's own weights are natively bf16 and training in fp16
    # instead causes a crash (see the GPU-detection cell's comments for the full explanation).
    bf16=USE_BF16,
    fp16=not USE_BF16,

    # gradient_checkpointing: normally, during the forward pass, the model keeps every
    # intermediate calculation in memory so it can use them later during backpropagation. Instead,
    # this setting throws most of them away and RECOMPUTES them when needed — trading extra
    # computation time for a large reduction in memory usage. USE_GRAD_CHECKPOINTING is set by the
    # GPU-detection cell (currently always on, as a safety margin).
    gradient_checkpointing=USE_GRAD_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False} if USE_GRAD_CHECKPOINTING else None,

    # max_length: the sequence-length cap explained in detail in the config cell above.
    max_length=MAX_SEQ_LENGTH,

    # assistant_only_loss: ensures the model is only "graded" (loss computed) on the tokens it
    # itself needs to generate — the answer — and NOT on the system prompt or the question, which
    # it never has to produce itself. Without this, the model would partially be trained to
    # predict its own instructions, which dilutes the learning signal for what we actually care
    # about (good answers).
    assistant_only_loss=True,

    # --- Experiment tracking: send live metrics to BOTH TensorBoard (local, no account needed)
    # AND Weights & Biases (cloud dashboard, needs the login from the cell near the top). Having
    # both means you can watch live in-notebook (TensorBoard) AND keep a permanent, shareable,
    # comparison-friendly record across runs (W&B) without extra effort.
    report_to=["tensorboard", "wandb"],
    logging_dir=f"{OUTPUT_DIR}/tb_logs",   # where TensorBoard's local log files go
    logging_strategy="steps",
    run_name=RUN_NAME,                      # the human-readable run label set in the config cell

    # load_best_model_at_end + metric_for_best_model: at the very end of training, instead of
    # keeping whichever checkpoint happened to be saved last, reload whichever checkpoint had the
    # LOWEST validation loss seen at any point during training. This protects against the case
    # where the model started overfitting near the end and its final state is actually worse than
    # an earlier one.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # loss_type="chunked_nll": computes the cross-entropy loss in small chunks instead of all at
    # once. Qwen3's vocabulary is ~150,000 possible tokens — computing the loss the "normal" way
    # would require building a [batch_size x sequence_length, 150000] tensor in one shot, which
    # is enormous and was the direct cause of an earlier out-of-memory crash in this notebook.
    # Chunking keeps memory usage flat regardless of sequence length.
    loss_type="chunked_nll",
)

import json
from transformers import TrainerCallback

class JSONLLoggerCallback(TrainerCallback):
    """A third logging destination, alongside TensorBoard and W&B: a plain local text file.

    Every time the trainer logs a metric (loss, eval_loss, learning_rate, etc. — the same
    numbers going to TensorBoard/W&B), this callback also appends it as one line of JSON to
    `training_log.jsonl`. This is useful because it's a plain file you can open, grep, or load
    with pandas at any time, completely independent of whether TensorBoard or W&B are working —
    a good fallback if either of those has a hiccup, and an easy way to keep a permanent copy
    since the Colab VM itself gets wiped on disconnect (W&B's cloud copy survives that too, but
    having a local file as well costs nothing).
    """
    def __init__(self, log_path):
        self.log_path = log_path
        open(self.log_path, "w").close()  # create/empty the file fresh at the start of this run

    def on_log(self, args, state, control, logs=None, **kwargs):
        # `on_log` is automatically called by the Trainer every time it has new metrics to report
        # (i.e. every LOGGING_STEPS steps, and at each evaluation). `logs` is a dict like
        # {"loss": 1.83, "learning_rate": 0.00019, ...} or {"eval_loss": 1.71} during evaluation.
        if logs is None:
            return
        record = dict(logs)
        record["step"] = state.global_step   # which training step this metric was recorded at
        record["epoch"] = state.epoch         # how many full passes over the data, as a fraction
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

TRAINING_LOG_PATH = f"{OUTPUT_DIR}/training_log.jsonl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=formatting_func,
    # peft_config (not a manually pre-wrapped model!): SFTTrainer applies the LoRA wrapping
    # itself when given the config this way, which is required for its internal memory-saving
    # patches (like chunked_nll above) to correctly recognize the model's structure. Wrapping the
    # model ourselves first caused a silent failure earlier in this notebook's development.
    peft_config=lora_config,
    callbacks=[JSONLLoggerCallback(TRAINING_LOG_PATH)],
)

# Sanity check: confirms LoRA is actually active and shows what fraction of the model's
# parameters are trainable. Expect well under 1% (typically 0.1-0.5%) — if this prints something
# close to 100%, LoRA did not get applied correctly and you'd effectively be full-fine-tuning,
# which would be far too slow/memory-hungry for this setup.
trainer.model.print_trainable_parameters()
print(f"W&B project: {WANDB_PROJECT}  |  Run name: {RUN_NAME}")
print(f"Training metrics will also be appended locally to: {TRAINING_LOG_PATH}")

### (Optional) Launch TensorBoard inline

Run this before or during training to watch loss curves live inside the notebook. It reads from
`logging_dir` (`{OUTPUT_DIR}/tb_logs`), which `SFTConfig`'s `report_to="tensorboard"` writes to automatically
— no extra account or API key needed, unlike Weights & Biases.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/tb_logs

### Diagnostic: confirm which precision mode is actually active

Run this once before `trainer.train()`. It checks every layer where an fp16 vs bf16 mismatch could be
hiding: the `ACCELERATE_MIXED_PRECISION` env var (which `TrainingArguments`/`SFTConfig` sets as a process-wide
side effect and which can persist oddly across runs in the same kernel), the resolved `accelerate` state, our
own config flags, and — most importantly — the actual dtype of every distinct parameter/buffer group in the
live model, since PEFT or `bitsandbytes` could be creating something in fp16 regardless of what we asked for.

In [ ]:
import os
from collections import Counter

print("--- Env var ---")
print("ACCELERATE_MIXED_PRECISION =", os.environ.get("ACCELERATE_MIXED_PRECISION", "<not set>"))

print("\n--- Our config flags ---")
print("USE_BF16 =", USE_BF16)
print("sft_config.bf16 =", sft_config.bf16, " sft_config.fp16 =", sft_config.fp16)

print("\n--- Live accelerate state (the actual source of truth for GradScaler) ---")
try:
    accel_state = trainer.accelerator.state
    print("accelerator.mixed_precision =", trainer.accelerator.mixed_precision)
    print("accelerator.scaler =", trainer.accelerator.scaler)
except Exception as e:
    print("Could not read trainer.accelerator state:", e)

print("\n--- Actual parameter/buffer dtypes in the live model (grouped by dtype) ---")
dtype_counts = Counter()
dtype_examples = {}
for name, p in trainer.model.named_parameters():
    dtype_counts[p.dtype] += 1
    if p.dtype not in dtype_examples:
        dtype_examples[p.dtype] = name
for dtype, count in dtype_counts.items():
    print(f"{dtype}: {count} params  (example: {dtype_examples[dtype]})")

print("\nIf you see torch.float16 anywhere above alongside bf16, or if "
      "ACCELERATE_MIXED_PRECISION says 'fp16', paste this whole output back.")

In [ ]:
import torch
torch.cuda.empty_cache()
trainer.train()

## 9. Save the LoRA adapter

In [ ]:
FINAL_ADAPTER_DIR = OUTPUT_DIR + "-final"
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print("Saved adapter + tokenizer to", FINAL_ADAPTER_DIR)
print("Trained on:", active_model_name)

# Explicitly close out the W&B run now that training + saving are done. Without this, the run
# can be left showing as "still running" on your wandb.ai dashboard until it eventually times
# out on its own — calling finish() marks it complete immediately and uploads any remaining
# buffered data.
import wandb
wandb.finish()

## 10. Quick qualitative test

In [ ]:
# Use trainer.model — the actual LoRA-wrapped, trained model. The bare `model` variable
# from Cell 5 is the unwrapped base model and was never trained (SFTTrainer applied the
# LoRA wrapping internally via peft_config=).
gen_model = trainer.model
gen_model.eval()
gen_model.config.use_cache = True  # was disabled for training (gradient checkpointing); re-enable for fast generation

test_example = eval_ds[0]["messages"][:2]  # system + user only, drop the gold answer
prompt_text = tokenizer.apply_chat_template(
    test_example, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(prompt_text, return_tensors="pt").to(gen_model.device)

with torch.no_grad():
    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("QUESTION CONTEXT:\n", test_example[1]["content"][:500], "...\n")
print("MODEL ANSWER:\n", generated, "\n")
print("GOLD ANSWER:\n", eval_ds[0]["messages"][2]["content"])

## 11. Review training logs

Reads back `training_log.jsonl` (written live by `JSONLLoggerCallback` during training) and plots train/eval
loss. This works even if the TensorBoard cell above was never opened, and gives you a plain file you can
download from Colab and keep alongside the model — useful since the Colab VM itself is wiped on disconnect.

In [ ]:
import json
import matplotlib.pyplot as plt

records = []
with open(TRAINING_LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} logged events from {TRAINING_LOG_PATH}")
print()

train_steps = [r["step"] for r in records if "loss" in r]
train_loss = [r["loss"] for r in records if "loss" in r]
eval_steps = [r["step"] for r in records if "eval_loss" in r]
eval_loss = [r["eval_loss"] for r in records if "eval_loss" in r]

if train_loss:
    plt.figure(figsize=(8, 4))
    plt.plot(train_steps, train_loss, label="train loss", marker="o", markersize=3)
    if eval_loss:
        plt.plot(eval_steps, eval_loss, label="eval loss", marker="s", markersize=4)
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(f"Training run: {active_model_name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("No training-loss events found yet — run trainer.train() first, or check "
          "LOGGING_STEPS isn't larger than your total step count.")

## Notes for this 6GB laptop GPU setup

This notebook already applies the settings that matter most for a 6GB card:
`LORA_TARGET_MODULES=["q_proj","v_proj"]`, `MAX_SEQ_LENGTH=384`, micro-batch `1` with gradient
checkpointing on, and `optim="paged_adamw_8bit"`.

If you still hit `CUDA out of memory`, try these in order:
1. **Switch to the 1.7B model**: set `MODEL_NAME = FALLBACK_MODEL_NAME` in the config cell and re-run from
   there — no other code changes needed.
2. **Drop `MAX_SEQ_LENGTH` further**, e.g. to 256, accepting more truncation on long examples.
3. **Drop `LORA_R` to 4** — halves the adapter's memory footprint at some cost to learning capacity.
4. **Close other GPU-using applications** (browser hardware acceleration, other notebooks/IDEs with GPU
   extensions) before running — on a 6GB card, 500MB of background usage is the difference between fitting
   and OOMing.
5. Windows users: if `bitsandbytes` 4-bit support is flaky on native Windows, run this under WSL2 instead.

Once `TRAIN_SUBSET_SIZE=200` finishes cleanly end-to-end (load → LoRA → train → save → generate), bump it up
or set it to `None` for the full 19,113-example training set, and restore `NUM_EPOCHS=2`. Expect a full run
on a 6GB laptop GPU to take meaningfully longer than on a cloud T4/L4 — this notebook checkpoints every
`SAVE_STEPS` steps, so if your session gets interrupted you can resume with
`trainer.train(resume_from_checkpoint=True)`.
